# Phase 1 – Zhang protocol replication

This notebook runs **two experiments** using the same model/dataset class:

1. **STRICT (leakage-safe)**: split by **GIF first**, then expand into 4-frame chunks.
2. **LEAKY (Zhang-like test)**: expand into chunks first, then random 80/20 split by **chunks** (this can leak chunks from the same GIF into train/test).

Run cells in order.

In [14]:

# =========================
# 0) Imports + safety
# =========================
import os, math, random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import models, transforms

from PIL import Image, ImageSequence, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True  # prevents crashes on truncated GIFs

from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

from tqdm.auto import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


device(type='cpu')

In [15]:

# =========================
# 1) Config (match Zhang Table 1)
# =========================
CSV_PATH = r"D:\IIT\Year 4\FYP\Datasets\GIFGIF_lucas\gifgif_emotion_labels.csv"   # <-- CHANGE
GIF_ROOT = None  # set to folder if CSV has relative paths, else keep None

Z_CLASSES = ["happiness","satisfaction","surprise","sadness","anger","disgust","fear"]

SEQ_LEN = 4
TARGET_SIZE = (224,224)
MAX_FRAMES_PER_GIF = 300

BATCH = 30
LR = 1e-4
WD = 1e-4
EPOCHS = 10

FREEZE_EPOCHS = 0  # Phase 1 reproduction: keep 0 (Zhang does not mention freezing)


In [16]:

# =========================
# 2) Load CSV + build Zhang-7 label from score columns (argmax over 7 scores)
# =========================
df = pd.read_csv(CSV_PATH)

if GIF_ROOT is not None:
    def _make_abs(p):
        if isinstance(p,str) and (os.path.isabs(p) or (len(p)>2 and p[1]==":")):
            return p
        return os.path.join(GIF_ROOT, str(p))
    df["gif_path"] = df["gif_path"].apply(_make_abs)

score_cols = [f"{c}_score" for c in Z_CLASSES]
missing = [c for c in score_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing score columns: {missing}")

df[score_cols] = df[score_cols].apply(pd.to_numeric, errors="coerce")
print("NaNs per score col:\n", df[score_cols].isna().sum())

df["label"] = df[score_cols].idxmax(axis=1).str.replace("_score","", regex=False)
df7 = df[df["label"].isin(Z_CLASSES)].copy()

print("\nAvailable counts (7-class):")
print(df7["label"].value_counts())
print("Total 7-class rows:", len(df7))


NaNs per score col:
 happiness_score       0
satisfaction_score    0
surprise_score        0
sadness_score         0
anger_score           0
disgust_score         0
fear_score            0
dtype: int64

Available counts (7-class):
label
happiness       2000
sadness          998
anger            911
surprise         679
satisfaction     652
disgust          459
fear             402
Name: count, dtype: int64
Total 7-class rows: 6101


In [17]:

# =========================
# 3) Zhang subset: first 300 per class => 2100 GIFs
# =========================
parts = []
for lab in Z_CLASSES:
    sub = df7[df7["label"] == lab]
    if len(sub) < 300:
        raise ValueError(f"Not enough samples for {lab}: {len(sub)} < 300")
    parts.append(sub.iloc[:300].copy())

df2100 = pd.concat(parts, ignore_index=True)

print("Total selected:", len(df2100))
print(df2100["label"].value_counts())


Total selected: 2100
label
happiness       300
satisfaction    300
surprise        300
sadness         300
anger           300
disgust         300
fear            300
Name: count, dtype: int64


In [18]:

# =========================
# 4) STRICT split: split by GIF (80/20 per class), then chunk
# =========================
train_parts, test_parts = [], []
for lab in Z_CLASSES:
    sub = df2100[df2100["label"] == lab]
    tr, te = train_test_split(sub, test_size=0.2, random_state=SEED, shuffle=True)
    train_parts.append(tr); test_parts.append(te)

train_df = pd.concat(train_parts, ignore_index=True)
test_df  = pd.concat(test_parts,  ignore_index=True)

print("Train:", len(train_df), "Test:", len(test_df))
print(train_df["label"].value_counts())


Train: 1680 Test: 420
label
happiness       240
satisfaction    240
surprise        240
sadness         240
anger           240
disgust         240
fear            240
Name: count, dtype: int64


In [19]:

# =========================
# 5) Transforms (paper-faithful: no augmentation)
# =========================
base_tf = transforms.Compose([
    transforms.Resize(TARGET_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])
train_tf = base_tf
test_tf  = base_tf

label2id = {lab:i for i,lab in enumerate(Z_CLASSES)}
id2label = {i:lab for lab,i in label2id.items()}


In [20]:

# =========================
# 6) GIF helpers + chunking
# =========================
def load_gif_frames_full(path, max_frames=300):
    frames = []
    im = Image.open(path)
    for i, fr in enumerate(ImageSequence.Iterator(im)):
        if max_frames is not None and i >= max_frames:
            break
        frames.append(fr.convert("RGB"))
    return frames

def chunk_consecutive(frames, seq_len=4):
    if len(frames) == 0:
        return [[Image.new("RGB", TARGET_SIZE, (0,0,0)) for _ in range(seq_len)]]
    chunks = []
    # non-overlapping consecutive chunks (matches the paper phrasing)
    for start in range(0, len(frames), seq_len):
        clip = frames[start:start+seq_len]
        if len(clip) < seq_len:
            # pad by repeating last frame
            clip = clip + [clip[-1]] * (seq_len - len(clip))
        chunks.append(clip)
    return chunks


In [21]:

# =========================
# 7) ONE Dataset class (used for BOTH strict and leaky experiments)
# =========================
class GifChunkDataset(Dataset):
    def __init__(self, df, transform, seq_len=4, max_frames=300):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.seq_len = seq_len
        self.max_frames = max_frames

        self.bad_gifs = 0
        self.index = []  # list of (row_idx, chunk_idx)

        for i in range(len(self.df)):
            path = self.df.loc[i, "gif_path"]
            try:
                frames = load_gif_frames_full(path, max_frames=self.max_frames)
                chunks = chunk_consecutive(frames, seq_len=self.seq_len)
                K = 2  # number of chunks per GIF (probe). Try 2 first. Later you can increase to 4 or "all".
                if len(chunks) <= K:
                    chosen = list(range(len(chunks)))
                else:
                    chosen = np.linspace(0, len(chunks) - 1, K).round().astype(int).tolist()

                for c in chosen:
                    self.index.append((i, c))

            except Exception:
                self.bad_gifs += 1
                self.index.append((i, 0))

        print(f"Built dataset: GIFs={len(self.df)}  Chunks={len(self.index)}  BadGIFs={self.bad_gifs}")

    def __len__(self):
        return len(self.index)

    def __getitem__(self, k):
        i, cidx = self.index[k]
        r = self.df.loc[i]
        lab = r["label"]
        path = r["gif_path"]

        try:
            frames = load_gif_frames_full(path, max_frames=self.max_frames)
            chunks = chunk_consecutive(frames, seq_len=self.seq_len)
            clip = chunks[min(cidx, len(chunks)-1)]
            x = torch.stack([self.transform(fr) for fr in clip], dim=0)  # [T,3,224,224]
        except Exception:
            x = torch.zeros((self.seq_len, 3, *TARGET_SIZE), dtype=torch.float32)

        y = label2id[lab]
        gif_id = r["gif_id"] if "gif_id" in r else os.path.basename(path)
        return x, y, gif_id


In [22]:

# =========================
# 8) Build STRICT datasets + loaders (GIF-split -> chunk)
# =========================
train_ds = GifChunkDataset(train_df, transform=train_tf, seq_len=SEQ_LEN, max_frames=MAX_FRAMES_PER_GIF)
test_ds  = GifChunkDataset(test_df,  transform=test_tf,  seq_len=SEQ_LEN, max_frames=MAX_FRAMES_PER_GIF)

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=0, pin_memory=(device.type=="cuda"))
test_loader  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False, num_workers=0, pin_memory=(device.type=="cuda"))

print("STRICT sanity:")
print("Train chunks:", len(train_ds), "Test chunks:", len(test_ds))
print("Bad train GIFs:", train_ds.bad_gifs, "Bad test GIFs:", test_ds.bad_gifs)

xb, yb, gid = next(iter(train_loader))
print("Batch:", xb.shape, yb.shape)


Built dataset: GIFs=1680  Chunks=3324  BadGIFs=0
Built dataset: GIFs=420  Chunks=836  BadGIFs=0
STRICT sanity:
Train chunks: 3324 Test chunks: 836
Bad train GIFs: 0 Bad test GIFs: 0
Batch: torch.Size([30, 4, 3, 224, 224]) torch.Size([30])


In [23]:

# =========================
# 9) Model: ResNet18 + ConvGRU (close to Zhang)
# =========================
class ConvGRUCell(nn.Module):
    def __init__(self, input_dim, hidden_dim, kernel_size=3):
        super().__init__()
        padding = kernel_size // 2
        self.hidden_dim = hidden_dim
        self.conv_z = nn.Conv2d(input_dim + hidden_dim, hidden_dim, kernel_size, padding=padding)
        self.conv_r = nn.Conv2d(input_dim + hidden_dim, hidden_dim, kernel_size, padding=padding)
        self.conv_h = nn.Conv2d(input_dim + hidden_dim, hidden_dim, kernel_size, padding=padding)

    def forward(self, x, h_prev):
        if h_prev is None:
            h_prev = torch.zeros(x.size(0), self.hidden_dim, x.size(2), x.size(3), device=x.device)
        cat = torch.cat([x, h_prev], dim=1)
        z = torch.sigmoid(self.conv_z(cat))
        r = torch.sigmoid(self.conv_r(cat))
        cat_r = torch.cat([x, r * h_prev], dim=1)
        h_hat = torch.tanh(self.conv_h(cat_r))
        h = (1 - z) * h_hat + z * h_prev
        return h

class ConvGRU(nn.Module):
    def __init__(self, input_dim, hidden_dim, kernel_size=3):
        super().__init__()
        self.cell = ConvGRUCell(input_dim, hidden_dim, kernel_size)

    def forward(self, x_seq):
        h = None
        for t in range(x_seq.size(1)):
            h = self.cell(x_seq[:, t], h)
        return h

class ResNetConvGRU(nn.Module):
    def __init__(self, num_classes=7, gru_hidden=256, dropout=0.4):
        super().__init__()
        base = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        self.backbone = nn.Sequential(*list(base.children())[:-2])  # [B,512,7,7]
        self.convgru = ConvGRU(input_dim=512, hidden_dim=gru_hidden, kernel_size=3)
        self.pool = nn.AdaptiveAvgPool2d((1,1))
        self.drop = nn.Dropout(p=dropout)
        self.fc = nn.Linear(gru_hidden, num_classes)

    def forward(self, x):
        feats = []
        for t in range(x.size(1)):
            feats.append(self.backbone(x[:, t]))
        feats = torch.stack(feats, dim=1)      # [B,T,512,7,7]
        h = self.convgru(feats)                # [B,H,7,7]
        v = self.pool(h).flatten(1)
        v = self.drop(v)
        return self.fc(v)

def init_model():
    m = ResNetConvGRU(num_classes=len(Z_CLASSES), gru_hidden=256, dropout=0.4).to(device)
    return m

model = init_model()
print("Params (M):", sum(p.numel() for p in model.parameters())/1e6)


Params (M): 16.487495


In [24]:

# =========================
# 10) Optimizer + train/eval helpers
# =========================
def make_optimizer(model):
    return torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WD)

def run_epoch(model, loader, opt=None):
    train = opt is not None
    model.train(train)
    losses = []
    y_true, y_pred = [], []

    for x, y, _gid in tqdm(loader, leave=False):
        x, y = x.to(device), y.to(device)

        if train:
            opt.zero_grad(set_to_none=True)

        logits = model(x)
        loss = F.cross_entropy(logits, y)

        if train:
            loss.backward()
            opt.step()

        losses.append(loss.item())
        y_true.extend(y.detach().cpu().numpy().tolist())
        y_pred.extend(logits.argmax(1).detach().cpu().numpy().tolist())

    acc = accuracy_score(y_true, y_pred)
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    return float(np.mean(losses)), float(acc), float(p), float(r), float(f1)


In [25]:

# =========================
# 11) TRAIN – STRICT (no leakage)
# =========================
model = init_model()
opt = make_optimizer(model)

history = []
CKPT_STRICT = "phase1_strict_best_acc.pt"

PATIENCE = 3
best_acc = -1.0
no_improve = 0

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc, tr_p, tr_r, tr_f1 = run_epoch(model, train_loader, opt=opt)
    te_loss, te_acc, te_p, te_r, te_f1 = run_epoch(model, test_loader,  opt=None)

    row = dict(
        epoch=epoch,
        train_loss=tr_loss, train_acc=tr_acc, train_p=tr_p, train_r=tr_r, train_f1=tr_f1,
        test_loss=te_loss,  test_acc=te_acc,  test_p=te_p,  test_r=te_r,  test_f1=te_f1,
        bad_train_gifs=train_ds.bad_gifs, bad_test_gifs=test_ds.bad_gifs,
        train_chunks=len(train_ds), test_chunks=len(test_ds),
    )
    history.append(row)
    print(row)

    improved = te_acc > best_acc + 1e-4

    if improved:
        best_acc = te_acc
        no_improve = 0
        torch.save({"model": model.state_dict(), "epoch": epoch, "metrics": row}, CKPT_STRICT)
        print("  💾 saved best ACC:", best_acc)
    else:
        no_improve += 1

    if no_improve >= PATIENCE:
        print(f"🛑 Early stopping: no improvement for {PATIENCE} epochs.")
        break

hist_df = pd.DataFrame(history)
hist_df.tail()


  0%|          | 0/111 [00:00<?, ?it/s]

  0%|          | 0/28 [00:00<?, ?it/s]

{'epoch': 1, 'train_loss': 1.866266461106034, 'train_acc': 0.24187725631768953, 'train_p': 0.24052930471500522, 'train_r': 0.24189260244793084, 'train_f1': 0.2400412341756219, 'test_loss': 1.889381114925657, 'test_acc': 0.23205741626794257, 'test_p': 0.23376507153074236, 'test_r': 0.23215591321274273, 'test_f1': 0.19474368298029585, 'bad_train_gifs': 0, 'bad_test_gifs': 0, 'train_chunks': 3324, 'test_chunks': 836}
  💾 saved best ACC: 0.23205741626794257


  0%|          | 0/111 [00:00<?, ?it/s]

  0%|          | 0/28 [00:00<?, ?it/s]

...

{'epoch': 3, 'train_loss': 0.19277486574273925, 'train_acc': 0.9596871239470517, 'train_p': 0.9596607964553237, 'train_r': 0.9596725227199114, 'train_f1': 0.9596522205828848, 'test_loss': 2.755173636334283, 'test_acc': 0.23086124401913877, 'test_p': 0.23647259375542568, 'test_r': 0.2310688682252562, 'test_f1': 0.22686014092691126, 'bad_train_gifs': 0, 'bad_test_gifs': 0, 'train_chunks': 3324, 'test_chunks': 836}


  0%|          | 0/111 [00:00<?, ?it/s]

  0%|          | 0/28 [00:00<?, ?it/s]

{'epoch': 4, 'train_loss': 0.07105358706804009, 'train_acc': 0.9873646209386282, 'train_p': 0.9873778878753464, 'train_r': 0.9873599473007655, 'train_f1': 0.9873626844102505, 'test_loss': 3.0948610731533597, 'test_acc': 0.21770334928229665, 'test_p': 0.2313178540953862, 'test_r': 0.2176207771244091, 'test_f1': 0.21560649755581124, 'bad_train_gifs': 0, 'bad_test_gifs': 0, 'train_chunks': 3324, 'test_chunks': 836}
🛑 Early stopping: no improvement for 3 epochs.


,epoch,train_loss,train_acc,train_p,train_r,train_f1,test_loss,test_acc,test_p,test_r,test_f1,bad_train_gifs,bad_test_gifs,train_chunks,test_chunks
0,1,1.866266,0.241877,0.240529,0.241893,0.240041,1.889381,0.232057,0.233765,0.232156,0.194744,0,0,3324,836
1,2,1.044521,0.653430,0.653421,0.653373,0.652697,2.363984,0.220096,0.245059,0.220154,0.216935,0,0,3324,836
2,3,0.192775,0.959687,0.959661,0.959673,0.959652,2.755174,0.230861,0.236473,0.231069,0.226860,0,0,3324,836
3,4,0.071054,0.987365,0.987378,0.987360,0.987363,3.094861,0.217703,0.231318,0.217621,0.215606,0,0,3324,836


In [26]:

# =========================
# 12) EVAL – STRICT: chunk-level + GIF-level aggregation
# =========================
from collections import defaultdict

ckpt = torch.load(CKPT_STRICT, map_location=device)
model = init_model()
model.load_state_dict(ckpt["model"])
model.eval()
print("Loaded:", CKPT_STRICT, "epoch:", ckpt["epoch"])
print("Snapshot:", ckpt["metrics"])

softmax = torch.nn.Softmax(dim=1)

# chunk-level
all_true, all_pred = [], []
with torch.no_grad():
    for x, y, gid in tqdm(test_loader):
        x = x.to(device)
        pred = model(x).argmax(1).cpu().numpy().tolist()
        all_pred.extend(pred); all_true.extend(y.numpy().tolist())

print("\n=== STRICT: Chunk-level ===")
print("Accuracy:", accuracy_score(all_true, all_pred))

# GIF-level aggregation (avg prob across chunks of the same GIF)
gif_probs = defaultdict(list)
gif_true  = {}

with torch.no_grad():
    for x, y, gid in tqdm(test_loader):
        x = x.to(device)
        probs = softmax(model(x)).cpu().numpy()
        y_np = y.numpy()
        for i in range(len(gid)):
            gif_probs[gid[i]].append(probs[i])
            gif_true[gid[i]] = int(y_np[i])

gif_y_true, gif_y_pred = [], []
for g, plist in gif_probs.items():
    avgp = np.mean(np.stack(plist, axis=0), axis=0)
    gif_y_true.append(gif_true[g])
    gif_y_pred.append(int(np.argmax(avgp)))

print("\n=== STRICT: GIF-level (avg softmax over chunks) ===")
print("Accuracy:", accuracy_score(gif_y_true, gif_y_pred))
print(classification_report(gif_y_true, gif_y_pred, target_names=Z_CLASSES, digits=4, zero_division=0))
print("Confusion Matrix:\n", confusion_matrix(gif_y_true, gif_y_pred))


Loaded: phase1_strict_best_acc.pt epoch: 1
Snapshot: {'epoch': 1, 'train_loss': 1.866266461106034, 'train_acc': 0.24187725631768953, 'train_p': 0.24052930471500522, 'train_r': 0.24189260244793084, 'train_f1': 0.2400412341756219, 'test_loss': 1.889381114925657, 'test_acc': 0.23205741626794257, 'test_p': 0.23376507153074236, 'test_r': 0.23215591321274273, 'test_f1': 0.19474368298029585, 'bad_train_gifs': 0, 'bad_test_gifs': 0, 'train_chunks': 3324, 'test_chunks': 836}


  0%|          | 0/28 [00:00<?, ?it/s]


=== STRICT: Chunk-level ===
Accuracy: 0.23205741626794257


  0%|          | 0/28 [00:00<?, ?it/s]


=== STRICT: GIF-level (avg softmax over chunks) ===
Accuracy: 0.23809523809523808
              precision    recall  f1-score   support

   happiness     0.1923    0.0833    0.1163        60
satisfaction     0.4000    0.0333    0.0615        60
    surprise     0.3448    0.1667    0.2247        60
     sadness     0.3871    0.2000    0.2637        60
       anger     0.1316    0.0833    0.1020        60
     disgust     0.2045    0.4500    0.2812        60
        fear     0.2453    0.6500    0.3562        60

    accuracy                         0.2381       420
   macro avg     0.2722    0.2381    0.2008       420
weighted avg     0.2722    0.2381    0.2008       420

Confusion Matrix:
 [[ 5  0  2  6  6 23 18]
 [ 3  2  3  1  8 21 22]
 [ 3  0 10  2  3 13 29]
 [ 1  2  2 12  7 19 17]
 [ 4  0  4  4  5 24 19]
 [ 4  1  3  5  5 27 15]
 [ 6  0  5  1  4  5 39]]


In [27]:

# =========================
# 13) LEAKY split: expand into chunks first, then random 80/20 split by chunks
# =========================
tmp_ds = GifChunkDataset(df2100, transform=test_tf, seq_len=SEQ_LEN, max_frames=MAX_FRAMES_PER_GIF)
print("All chunks total:", len(tmp_ds), "BadGIFs:", tmp_ds.bad_gifs)

all_idx = np.arange(len(tmp_ds))
tr_idx, te_idx = train_test_split(all_idx, test_size=0.2, random_state=SEED, shuffle=True)

leaky_train_ds = Subset(tmp_ds, tr_idx)
leaky_test_ds  = Subset(tmp_ds, te_idx)

leaky_train_loader = DataLoader(leaky_train_ds, batch_size=BATCH, shuffle=True,  num_workers=0, pin_memory=(device.type=="cuda"))
leaky_test_loader  = DataLoader(leaky_test_ds,  batch_size=BATCH, shuffle=False, num_workers=0, pin_memory=(device.type=="cuda"))

print("Leaky train chunks:", len(leaky_train_ds), "Leaky test chunks:", len(leaky_test_ds))


Built dataset: GIFs=2100  Chunks=4160  BadGIFs=0
All chunks total: 4160 BadGIFs: 0
Leaky train chunks: 3328 Leaky test chunks: 832


In [28]:
# =========================
# 14) TRAIN – LEAKY (Zhang-like; may inflate results)
# =========================
model = init_model()
opt = make_optimizer(model)

history_leaky = []

CKPT_LEAKY = "phase1_leaky_best_acc.pt"
PATIENCE = 3
best_acc = -1.0
no_improve = 0

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc, tr_p, tr_r, tr_f1 = run_epoch(model, leaky_train_loader, opt=opt)
    te_loss, te_acc, te_p, te_r, te_f1 = run_epoch(model, leaky_test_loader,  opt=None)

    row = dict(
        epoch=epoch,
        train_loss=tr_loss, train_acc=tr_acc, train_p=tr_p, train_r=tr_r, train_f1=tr_f1,
        test_loss=te_loss,  test_acc=te_acc,  test_p=te_p,  test_r=te_r,  test_f1=te_f1,
        total_chunks=len(tmp_ds),
        leaky_train_chunks=len(leaky_train_ds),
        leaky_test_chunks=len(leaky_test_ds),
    )
    history_leaky.append(row)
    print(row)

    improved = te_acc > best_acc + 1e-4
    if improved:
        best_acc = te_acc
        no_improve = 0
        torch.save({"model": model.state_dict(), "epoch": epoch, "metrics": row}, CKPT_LEAKY)
        print("  💾 saved best ACC:", best_acc)
    else:
        no_improve += 1

    if no_improve >= PATIENCE:
        print(f"🛑 Early stopping: no improvement for {PATIENCE} epochs.")
        break

hist_df_leaky = pd.DataFrame(history_leaky)
hist_df_leaky.tail()


  0%|          | 0/111 [00:00<?, ?it/s]

  0%|          | 0/28 [00:00<?, ?it/s]

{'epoch': 1, 'train_loss': 1.8944786067481514, 'train_acc': 0.2205528846153846, 'train_p': 0.217573232686686, 'train_r': 0.21964084905018577, 'train_f1': 0.21680513206574534, 'test_loss': 1.7689260372093745, 'test_acc': 0.3173076923076923, 'test_p': 0.3658209651599266, 'test_r': 0.32693686977355424, 'test_f1': 0.29152022508315234, 'total_chunks': 4160, 'leaky_train_chunks': 3328, 'leaky_test_chunks': 832}
  💾 saved best ACC: 0.3173076923076923


  0%|          | 0/111 [00:00<?, ?it/s]

  0%|          | 0/28 [00:00<?, ?it/s]

...

{'epoch': 7, 'train_loss': 0.12212538383565508, 'train_acc': 0.9651442307692307, 'train_p': 0.9651181351148344, 'train_r': 0.9651764711263964, 'train_f1': 0.9651350664865517, 'test_loss': 1.4294944831303187, 'test_acc': 0.59375, 'test_p': 0.6160917904719075, 'test_r': 0.6000515241070108, 'test_f1': 0.5977401652229625, 'total_chunks': 4160, 'leaky_train_chunks': 3328, 'leaky_test_chunks': 832}


  0%|          | 0/111 [00:00<?, ?it/s]

  0%|          | 0/28 [00:00<?, ?it/s]

{'epoch': 8, 'train_loss': 0.12364484842128314, 'train_acc': 0.9633413461538461, 'train_p': 0.9633762366219321, 'train_r': 0.9634869000336479, 'train_f1': 0.9634091421481913, 'test_loss': 1.4580946415662766, 'test_acc': 0.6057692307692307, 'test_p': 0.6135464926792634, 'test_r': 0.6112430221198686, 'test_f1': 0.6084767860449202, 'total_chunks': 4160, 'leaky_train_chunks': 3328, 'leaky_test_chunks': 832}
🛑 Early stopping: no improvement for 3 epochs.


,epoch,train_loss,train_acc,train_p,train_r,train_f1,test_loss,test_acc,test_p,test_r,test_f1,total_chunks,leaky_train_chunks,leaky_test_chunks
3,4,0.082899,0.982873,0.982869,0.982785,0.982816,1.264657,0.637019,0.672990,0.638381,0.644301,4160,3328,832
4,5,0.045249,0.990084,0.990030,0.990059,0.990027,1.261691,0.650240,0.666863,0.656699,0.652379,4160,3328,832
5,6,0.048711,0.986178,0.986218,0.986220,0.986217,1.393969,0.618990,0.640225,0.621675,0.620676,4160,3328,832
6,7,0.122125,0.965144,0.965118,0.965176,0.965135,1.429494,0.593750,0.616092,0.600052,0.597740,4160,3328,832
7,8,0.123645,0.963341,0.963376,0.963487,0.963409,1.458095,0.605769,0.613546,0.611243,0.608477,4160,3328,832


In [29]:

# =========================
# 15) EVAL – LEAKY: chunk-level only
# =========================
ckpt = torch.load(CKPT_LEAKY, map_location=device)
model = init_model()
model.load_state_dict(ckpt["model"])
model.eval()
print("Loaded:", CKPT_LEAKY, "epoch:", ckpt["epoch"])
print("Snapshot:", ckpt["metrics"])

all_true, all_pred = [], []
with torch.no_grad():
    for x, y, gid in tqdm(leaky_test_loader):
        x = x.to(device)
        pred = model(x).argmax(1).cpu().numpy().tolist()
        all_pred.extend(pred); all_true.extend(y.numpy().tolist())

print("\n=== LEAKY: Chunk-level ===")
print("Accuracy:", accuracy_score(all_true, all_pred))
print(classification_report(all_true, all_pred, target_names=Z_CLASSES, digits=4, zero_division=0))
print("Confusion Matrix:\n", confusion_matrix(all_true, all_pred))


Loaded: phase1_leaky_best_acc.pt epoch: 5
Snapshot: {'epoch': 5, 'train_loss': 0.04524867162779645, 'train_acc': 0.9900841346153846, 'train_p': 0.9900301442633108, 'train_r': 0.9900593162913321, 'train_f1': 0.9900268265540675, 'test_loss': 1.2616911296333586, 'test_acc': 0.6502403846153846, 'test_p': 0.6668632892614966, 'test_r': 0.6566991673860142, 'test_f1': 0.6523789251396629, 'total_chunks': 4160, 'leaky_train_chunks': 3328, 'leaky_test_chunks': 832}


  0%|          | 0/28 [00:00<?, ?it/s]


=== LEAKY: Chunk-level ===
Accuracy: 0.6502403846153846
              precision    recall  f1-score   support

   happiness     0.7222    0.5612    0.6316       139
satisfaction     0.5867    0.7333    0.6519       120
    surprise     0.5414    0.6538    0.5923       130
     sadness     0.6937    0.7938    0.7404        97
       anger     0.6341    0.7091    0.6695       110
     disgust     0.7091    0.6667    0.6872       117
        fear     0.7808    0.4790    0.5938       119

    accuracy                         0.6502       832
   macro avg     0.6669    0.6567    0.6524       832
weighted avg     0.6660    0.6502    0.6485       832

Confusion Matrix:
 [[78 23 19  5  5  8  1]
 [ 7 88  4  4 11  3  3]
 [10  8 85  9  7  7  4]
 [ 1  3  5 77  3  6  2]
 [ 3  6 13  4 78  2  4]
 [ 0 11 10  4 12 78  2]
 [ 9 11 21  8  7  6 57]]
